# Exploration Notebook
## Letterboxd Dashboard
### Reid B.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
ratings   = pd.read_csv('data/ratings.csv')
diary     = pd.read_csv('data/diary.csv')
watched   = pd.read_csv('data/watched.csv')
watchlist = pd.read_csv('data/watchlist.csv')

for df in [ratings, diary, watched, watchlist]:
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    if 'year' in df.columns:
        df['year'] = df['year'].astype('Int64')

diary['watched_date'] = pd.to_datetime(diary['watched_date'], errors='coerce')
diary['date']         = pd.to_datetime(diary['date'], errors='coerce')
diary_dated           = diary.dropna(subset=['watched_date'])
ratings['decade']     = (ratings['year'] // 10 * 10).astype(str) + 's'

print(f'Ratings: {ratings.shape}')
print(f'Diary:   {diary.shape}')

## 1. Rating Distribution

In [ ]:
rc = ratings['rating'].value_counts().sort_index()

fig, ax = plt.subplots()
bars = ax.bar(rc.index.astype(str), rc.values, color='coral', edgecolor='white')

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('My rating distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Rating (★)')
ax.set_ylabel('Films')
plt.tight_layout()
plt.show()

## 2. Most Watched Release Years

In [ ]:
year_counts = ratings['year'].value_counts().sort_index()

fig, ax = plt.subplots()
ax.bar(year_counts.index.astype(str), year_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Films watched by release year', fontsize=14, fontweight='bold')
ax.set_xlabel('Release year')
ax.set_ylabel('Films')
plt.xticks(rotation=45, ha='right')
# show every 5th label to avoid crowding
for i, label in enumerate(ax.get_xticklabels()):
    if i % 5 != 0:
        label.set_visible(False)
plt.tight_layout()
plt.show()

## 3. Do I Rate Older Films Higher?
Filtered to years with at least 3 films. Bubble size = number of films watched.

In [ ]:
year_ratings = (
    ratings.groupby('year')['rating']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_rating', 'count': 'films'})
    .query('films >= 3')
    .sort_values('year')
)

fig, ax = plt.subplots()
scatter = ax.scatter(
    year_ratings['year'],
    year_ratings['avg_rating'],
    s=year_ratings['films'] * 20,
    c=year_ratings['avg_rating'],
    cmap='RdYlGn',
    alpha=0.7,
    edgecolors='white',
    linewidths=0.5
)
plt.colorbar(scatter, ax=ax, label='Avg rating')
ax.set_title('Avg rating by release year (min 3 films)', fontsize=14, fontweight='bold')
ax.set_xlabel('Release year')
ax.set_ylabel('Avg rating')
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

## 4. Average Rating by Decade
Filtered to decades with at least 5 films.

In [ ]:
decade_stats = (
    ratings.groupby('decade')['rating']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_rating', 'count': 'films'})
    .query('films >= 5')
    .sort_values('decade')
)

fig, ax = plt.subplots()
bars = ax.bar(decade_stats['decade'], decade_stats['avg_rating'].round(2),
              color='mediumpurple', edgecolor='white')

for bar, val in zip(bars, decade_stats['avg_rating']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f'{val:.2f}',
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Avg rating by decade (min 5 films)', fontsize=14, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Avg rating')
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.show()

## 5. Films Logged Per Month

In [ ]:
monthly = (
    diary_dated
    .groupby(diary_dated['watched_date'].dt.to_period('M'))
    .size()
    .reset_index(name='films')
)
monthly['watched_date'] = monthly['watched_date'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['watched_date'], monthly['films'],
        color='#e63946', linewidth=2, marker='o', markersize=4)
ax.fill_between(range(len(monthly)), monthly['films'], alpha=0.15, color='#e63946')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['watched_date'], rotation=45, ha='right')
# show every 3rd label
for i, label in enumerate(ax.get_xticklabels()):
    if i % 3 != 0:
        label.set_visible(False)
ax.set_title('Films watched per month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Films watched')
plt.tight_layout()
plt.show()

## 6. Films by Decade (Count)

In [ ]:
decade_counts = ratings['decade'].value_counts().sort_index()

fig, ax = plt.subplots()
bars = ax.bar(decade_counts.index, decade_counts.values, color='teal', edgecolor='white')

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Films watched by decade', fontsize=14, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Films')
plt.tight_layout()
plt.show()

## 7. Most Rewatched Films

In [ ]:
rewatched = (
    diary.groupby(['name', 'year'])
    .size()
    .reset_index(name='watches')
    .query('watches > 1')
    .sort_values('watches')
    .tail(15)
)
rewatched['label'] = rewatched['name'] + ' (' + rewatched['year'].astype(str) + ')'

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(rewatched['label'], rewatched['watches'], color='#e63946', edgecolor='white')

for bar in bars:
    ax.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        str(int(bar.get_width())),
        va='center', fontsize=9
    )

ax.set_title('Most rewatched films', fontsize=14, fontweight='bold')
ax.set_xlabel('Times watched')
plt.tight_layout()
plt.show()

## 8. Films Watched Per Calendar Year

In [ ]:
per_year = (
    diary_dated
    .groupby(diary_dated['watched_date'].dt.year)
    .size()
    .reset_index(name='films')
    .rename(columns={'watched_date': 'year'})
)

fig, ax = plt.subplots()
bars = ax.bar(per_year['year'].astype(str), per_year['films'],
              color='mediumseagreen', edgecolor='white')

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(int(bar.get_height())),
        ha='center', va='bottom', fontsize=9
    )

ax.set_title('Films watched per calendar year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Films watched')
plt.tight_layout()
plt.show()